# Gold Layer Notebook

**Purpose:** Create business-ready Gold tables from the cleaned Silver titles dataset for dashboard reporting and stakeholder analysis.

This notebook creates business-ready Gold Delta tables from the cleaned Netflix Silver data stored in ADLS Gen2.

**Gold outputs**
- `gold_content_by_type`
- `gold_titles_by_release_year`
- `gold_titles_by_release_year_trend`
- `gold_titles_by_rating`
- `gold_top20_release_years`

## Gold Table: `gold_titles_by_release_year`

**Business question:** How many titles were released in each year across the full catalog?

In [ ]:
gold_titles_by_release_year = (
    df_titles
    .filter(col("release_year").isNotNull())
    .groupBy("release_year")
    .agg(count("*").alias("total_titles"))
    .orderBy("release_year")
)

display(gold_titles_by_release_year)

gold_titles_by_release_year.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_titles_by_release_year")

## Gold Table: `gold_top20_release_years`

**Business question:** Which release years have the highest number of Netflix titles?

In [ ]:
gold_top20_release_years = (
    gold_titles_by_release_year
    .orderBy(desc("total_titles"))
    .limit(20)
)

display(gold_top20_release_years)

gold_top20_release_years.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_top20_release_years")

## 1. Environment Setup

Configure the Unity Catalog destination for Gold tables.

In [0]:
from pyspark.sql.functions import *

CATALOG = "netflix_adb_luc"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")

print("Gold schema is ready.")

Gold schema is ready.


## 2. Read Silver Source Data

Source: `abfss://silver@nextflixprojectdltluc.dfs.core.windows.net/netflix_titles`

This Silver Delta table is used as the source for all Gold aggregations.

In [0]:
silver_titles_path = "abfss://silver@nextflixprojectdltluc.dfs.core.windows.net/netflix_titles"

df_titles = spark.read.format("delta").load(silver_titles_path)

display(df_titles.limit(10))
df_titles.printSchema()

duration_minutes,duration_seasons,type,title,date_added,release_year,rating,description,show_id,_rescued_data,Shorttile,type_flag,duration_ranking
312,1,Movie,Black Mirror: Bandersnatch,12/28/2018,2018,TV,"In 1984, a young programmer begins to question reality as he adapts a dark fantasy novel into a video game. A mind-bending tale with multiple endings.",80988062,null,Black Mirror,1,1
228,1,Movie,Sangam,12/31/2019,1964,TV,"Returning home from war after being assumed dead, a pilot weds the woman he has long loved, unaware that she had been planning to marry his best friend.",60002818,null,Sangam,1,2
224,1,Movie,Lagaan,12/8/2017,2001,PG,"In 1890s India, an arrogant British commander challenges the harshly taxed residents of Champaner to a high-stakes cricket match.",60020906,null,Lagaan,1,3
214,1,Movie,Jodhaa Akbar,10/1/2018,2008,TV,"In 16th-century India, what begins as a strategic alliance between a Mughal emperor and a Hindu princess becomes a genuine opportunity for true love.",70090035,null,Jodhaa Akbar,1,4
209,1,Movie,The Irishman,11/27/2019,2019,R,Hit man Frank Sheeran looks back at the secrets he kept as a loyal member of the Bufalino crime family in this acclaimed film from Martin Scorsese.,80175798,null,The Irishman,1,5
205,1,Movie,The Gospel of Luke,10/19/2018,2015,TV,Word-for-word Bible texts of the entire book of Luke are narrated and re-enacted in this epic production of the Gospel's accounts of Jesus's life.,81035749,null,The Gospel of Luke,1,6
203,1,Movie,What's Your Raashee?,8/15/2018,2009,TV,"To protect his family from ruin, Yogesh must marry his dream girl in only ten days, so he rushes into dating women with different astrological signs.",70123118,null,What's Your Raashee?,1,7
201,1,Movie,The Lord of the Rings: The Return of the King,1/1/2020,2003,PG,"Aragorn is revealed as the heir to the ancient kings as he, Gandalf and the other members of the broken fellowship struggle to save Gondor.",60004484,null,The Lord of the Rings,1,8
200,1,Movie,Doctor Zhivago,11/1/2019,1965,PG,A young physician and his beautiful mistress get swept up in the danger and drama of the Bolshevik Revolution in this Oscar-winning epic.,449931,null,Doctor Zhivago,1,9
196,1,Movie,Elephants Dream 4 Hour,8/23/2018,2006,TV,"Friends Proog and Emo live in a capricious, seemingly infinite machine with a sinister purpose in this experimental computer-animated short.",70274390,null,Elephants Dream 4 Hour,1,10


root
 |-- duration_minutes: integer (nullable = true)
 |-- duration_seasons: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- description: string (nullable = true)
 |-- show_id: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- Shorttile: string (nullable = true)
 |-- type_flag: integer (nullable = true)
 |-- duration_ranking: integer (nullable = true)



## Gold Table: `gold_content_by_type`

**Business question:** What is the distribution of Netflix content by type?

In [0]:
gold_content_by_type = (
    df_titles
    .filter(col("type").isin("Movie", "TV Show"))
    .groupBy("type")
    .agg(count("*").alias("total_titles"))
    .orderBy(desc("total_titles"))
)

display(gold_content_by_type)

gold_content_by_type.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_content_by_type")

type,total_titles
Movie,4265
TV Show,1969


Databricks visualization. Run in Databricks to view.

## Gold Table: `gold_titles_by_release_year_trend`

**Business question:** How has Netflix title release volume changed over recent years?

In [0]:
gold_titles_by_release_year_trend = (
    df_titles
    .filter(col("release_year").isNotNull())
    .filter((col("release_year") >= 2001) & (col("release_year") <= 2020))
    .groupBy("release_year")
    .agg(count("*").alias("total_titles"))
    .orderBy("release_year")
)

display(gold_titles_by_release_year_trend)

gold_titles_by_release_year_trend.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_titles_by_release_year_trend")

release_year,total_titles
2001,34
2002,38
2003,43
2004,49
2005,63
2006,68
2007,71
2008,107
2009,121
2010,149


Databricks visualization. Run in Databricks to view.

## Gold Table: `gold_titles_by_rating`

**Business question:** Which Netflix content ratings appear most frequently?

In [0]:
valid_ratings = [
    "G", "PG", "PG-13", "R", "NC-17",
    "TV-Y", "TV-Y7", "TV-G", "TV-PG", "TV-14", "TV-MA",
    "NR", "UR"
]

gold_titles_by_rating = (
    df_titles
    .filter(col("rating").isNotNull())
    .filter(col("rating").isin(valid_ratings))
    .groupBy("rating")
    .agg(count("*").alias("total_titles"))
    .orderBy(desc("total_titles"))
)

display(gold_titles_by_rating)

gold_titles_by_rating.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_titles_by_rating")

rating,total_titles
R,508
PG,470
NR,218
G,37
UR,7


Databricks visualization. Run in Databricks to view.

## 3. Validate Gold Tables

Confirm that all Gold tables were created in Unity Catalog.

In [0]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}"))

database,tableName,isTemporary
gold,gold_content_by_type,false
gold,gold_titles_by_rating,false
gold,gold_titles_by_release_year,false
gold,gold_titles_by_release_year_trend,false
gold,gold_top20_release_years,false


## 4. Gold Layer Summary

The Gold layer now contains curated analytical tables for content type distribution, release-year trend analysis, and rating distribution. These tables can be queried directly from Unity Catalog or connected to downstream BI tools.